<a href="https://colab.research.google.com/github/Rabab-kh/GEMMINI-API/blob/main/Re%C3%A7uFiscale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install google-genai pydantic

In [10]:
import os
from google.colab import userdata
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

In [11]:
# 1. Récupération de la clé API Google AI Studio depuis l'onglet 🔑
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

In [12]:
# Initialisation du client Google GenAI
client = genai.Client()

In [13]:
# 2. Définition de la structure Pydantic
class RecuFiscal(BaseModel):
    marchand: str = Field(description="Nom du magasin ou de l'entreprise")
    date: str = Field(description="Date de l'achat au format AAAA-MM-JJ")
    montant_total: float = Field(description="Le montant total payé (nombre décimal)")
    devise: str = Field(description="Le symbole ou code de la devise (ex: EUR, USD, MAD)")
    articles: list[str] = Field(description="Liste concise des articles achetés")

In [20]:
# 3. Fonction d'extraction mise à jour
def extraire_donnees_recu_gemini(texte_brut_ticket: str) -> RecuFiscal:
    prompt_systeme = """
    Tu es un assistant de comptabilité expert. Ton rôle est d'extraire les données clés d'un texte brut issu d'un ticket de caisse.
    Insiste bien sur le format de date AAAA-MM-JJ.
    """

    response = client.models.generate_content(
        model='gemini-2.5-flash', # Version à jour et standard du SDK actuel
        contents=texte_brut_ticket,
        config=types.GenerateContentConfig(
            system_instruction=prompt_systeme,
            response_mime_type="application/json",
            response_schema=RecuFiscal,
        ),
    )

    return RecuFiscal.model_validate_json(response.text)

In [21]:
# --- ZONE DE TEST ---
ticket_sale = " CAFE DES AMIS  \n Date: 28 Fevrier 2026 \n 2x Espresso -- 40.00 \n 1x Croissant -- 15.00 \n------------------------\n TOTAL NET A PAYER: 55.00 MAD \n Merci de votre visite !"

In [22]:
donnees_propres = extraire_donnees_recu_gemini(ticket_sale)

In [23]:
print(donnees_propres)

marchand='CAFE DES AMIS' date='2026-02-28' montant_total=55.0 devise='MAD' articles=['Espresso', 'Croissant']
